In [1]:
%load_ext autoreload
%autoreload 2

# Walkthrough Example - GIS functionalities

This notebook provides a practical walkthrough of the `gis` subpackage of `oceanicospy`. It covers reading, writing, reprojecting, merging, masking,and visualizing XYZ point cloud data for oceanographic and coastal applications. All examples follow a single dataset: a topobathymetric survey of a coastal area, composed of a high-resolution bathymetric tile (10 m, EPSG:9377) and a lower-resolution tile (50 m, EPSG:32617) stored as a point shapefile. Together they illustrate a complete preprocessing pipeline from raw input files to a merged, analysis-ready point cloud.

```{hint}
To replicate the examples in this notebook you will need:
- `oceanicospy` installed in your Python environment.
- Two point data files in your working directory:
  - A headerless XYZ text file.
  - A point shapefile (.shp).

For the merging section to be meaningful, the two files should cover overlapping areas. If they do not overlap, the merge will simply
concatenate the data without resolving any conflicts.

Replace the file paths in the cells below with the paths to your own files. The workflow generalizes to any XYZ or point vector file.
```

# Importing libraries

The following libraries are imported to support the examples in this notebook.

In [2]:
import pandas as pd
import geopandas as gpd

The different modules in the `oceanicospy.gis` subpackage are imported to showcase the gis capabilities.

In [3]:
from oceanicospy.gis import (
    XYZFormatSpec,
    infer_xyz_format,
    read_xyz,
    write_xyz,
    load_xyz_as_geodataframe,
    save_xyz_from_geodataframe,
    PointFileReprojector,
    reproject_xyz_file,
)

# XYZ file I/O

The `gis` subpackage provides a set of functions to read and write XYZ plain-text files. All of them accept an `XYZFormatSpec` object that describes the on-disk layout of the file: column delimiter, whether a header row is present, and column names for X, Y and Z.

`XYZFormatSpec` describes how an XYZ file is structured on disk. When the format is known in advance, it is good practice to define it explicitly rather than relying on automatic detection. This ensures consistent behaviour across reading and writing operations.

In [4]:
spec = XYZFormatSpec(
    delimiter=" ",
    has_header=False,
    x_column="x",
    y_column="y",
    z_column="z",
)
spec

XYZFormatSpec(delimiter=' ', has_header=False, x_column='x', y_column='y', z_column='z', encoding='utf-8')

When the file format is not known in advance, `infer_xyz_format` inspects a sample of lines and returns an `XYZFormatSpec` with the detected settings. This is useful for exploratory work or when handling files from different sources.

In [5]:
xyz_path = "../data/gis/bat_sai_10m_9377.xyz"

detected_spec = infer_xyz_format(xyz_path)
detected_spec

XYZFormatSpec(delimiter=' ', has_header=False, x_column='x', y_column='y', z_column='z', encoding='utf-8')

`read_xyz` loads an XYZ file into a `pandas.DataFrame`. When a `format_spec` is provided the file is parsed accordingly. If omitted, the format is detected automatically.

In [6]:
df = read_xyz(xyz_path, format_spec=detected_spec)
df.head()

,x,y,z
0,4051049.061,2959341.130,-10.00
1,4051059.163,2959340.821,-10.00
2,4051069.266,2959340.512,-10.00
3,4051079.368,2959340.203,-8.87
4,4051089.470,2959339.895,-6.70


A quick inspection of the Z column shows the value range. In this example the dataset contains bathymetry only, so negative Z values correspond to points that fall on land and are excluded. If your dataset is topobathymetric, negative values represent land elevation and should be kept.

In [7]:
df["z"].describe()

count    600000.000000
mean        125.659815
std         182.689073
min         -10.000000
25%           2.920000
50%          20.924000
75%         226.783250
max         660.102000
Name: z, dtype: float64

In [8]:
df_bathy = df[df["z"] >= 0].reset_index(drop=True)
df_bathy["z"].describe()

count    547797.000000
mean        138.491334
std         186.179717
min           0.000000
25%           4.720000
50%          31.268000
75%         266.826000
max         660.102000
Name: z, dtype: float64

`write_xyz` saves a `pandas.DataFrame` to an XYZ plain-text file. The filtered DataFrame is saved here for use in the following sections.

In [9]:
bathy_path = "../data/gis/bat_sai_10m_9377-filt.xyz"

write_xyz(df_bathy, bathy_path, format_spec=detected_spec)

`load_xyz_as_geodataframe` reads an XYZ file and returns a `geopandas.GeoDataFrame` with Point geometries built from the X and Y columns. The CRS must be provided since XYZ files carry no spatial metadata. No reprojection is performed at this stage.

In [ ]:
gdf = load_xyz_as_geodataframe(bathy_path, format_spec=detected_spec, crs=9377)
gdf.head()

# Reprojecting point files

The `crs_tools` module provides tools to reproject point data between coordinate reference systems. Both vector files and XYZ text files are
supported. All EPSG codes can be passed as integers (e.g. `9377`) or strings (e.g. `"EPSG:9377"`).

`PointFileReprojector` loads a point file, reprojects it to a target CRS, and exports the result as an XYZ file. The same `format_spec` is used for both reading and writing, so input and output share the same layout by default.

## From a vector file

`PointFileReprojector` loads the point shapefile and reads its CRS automatically from the embedded `.prj` file. If the file has no embedded
CRS, the source CRS can be specified manually via `source_epsg`. The data is then reprojected to the target CRS and exported as an XYZ file, ready to be used alongside the bathymetric tile in the following sections.

Before loading, it is recommended to inspect the shapefile column names to confirm the correct `z_column` name, as it may vary depending on the source software.

In [14]:
shp_path = "../data/gis/bat_sai_50m_32617.shp"
gdf_preview = gpd.read_file(shp_path)
print(gdf_preview.columns.tolist())
print(gdf_preview.crs)

['field_1', 'field_2', 'field_3', 'geometry']
EPSG:32617


In [15]:
reprojector = PointFileReprojector(
    input_path=shp_path,
    z_column="field_3",
)

print(reprojector.crs)

EPSG:32617


In [17]:
reprojector.reproject_to_epsg(32617)
print(reprojector.crs)

EPSG:32617


Once reprojected, the result is exported as an XYZ file using `to_xyz`, which internally delegates to `save_xyz_from_geodataframe` from the `io_xyz` module. The output is a plain-text file ready for the merging step.

In [18]:
reprojected_path = "../data/gis/bat_sai_50m_32617.xyz"
reprojector.to_xyz(reprojected_path)

In [20]:
reproject_xyz_file(
    input_path=shp_path,
    output_path="../data/gis/bat_sai_50m_9377.xyz",
    source_epsg=32617,
    target_epsg=9377,
    z_column="field_3"
)

In [21]:
import geopandas as gpd

shp_path = "../data/gis/bat_sai_50m_32617.shp"
gdf = gpd.read_file(shp_path)

print("CRS original:", gdf.crs)
print("Columnas:", gdf.columns.tolist())
print("Primeras coordenadas:", gdf.geometry.iloc[0])

gdf_reprojected = gdf.to_crs("EPSG:9377")
print("CRS reproyectado:", gdf_reprojected.crs)
print("Primeras coordenadas reproyectadas:", gdf_reprojected.geometry.iloc[0])

CRS original: EPSG:32617
Columnas: ['field_1', 'field_2', 'field_3', 'geometry']
Primeras coordenadas: POINT Z (-3010996.715132651 -166428.51633077709 1927.7127497954)
CRS reproyectado: EPSG:9377
Primeras coordenadas reproyectadas: POINT Z (413380.00000000186 1375010.000000001 1927.7127497954)
